# Milestone 3 — P2.Data Preprocessing Pipeline

`1) Feature Engineering → 2) Feature Selection → 3) Encoding → 4) Train/Test Split → 5) Scaling (fit on training set only) → 6) Final Check → 7) Saving Final Files`

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('C:\\Users\\user\\Downloads\\milestone3\\AI4I_with_synthetic_data_cleaned_data.csv')  
target = 'Machine failure'
print('Shape BEFORE preprocessing:', df.shape)
cols_before = df.shape[1]
text_cols_before = df.select_dtypes(exclude='number').columns.tolist()
print('Non-numeric columns:', text_cols_before)

Shape BEFORE preprocessing: (12500, 14)
Non-numeric columns: ['Product ID', 'Type']


## Step 1 — Feature Engineering 



In [2]:
df['Temp_Diff']          = df['Process temperature [K]'] - df['Air temperature [K]']
df['Mechanical_Power_W'] = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * (2 * np.pi / 60)
df['Strain_Index']       = df['Torque [Nm]'] * df['Tool wear [min]']

new_feats = ['Temp_Diff', 'Mechanical_Power_W', 'Strain_Index']
print('Correlation of new features with the target:')
print(df[new_feats + [target]].corr()[target].drop(target).round(2).to_string())
print('\nCorr(Power, Torque) =', round(df['Mechanical_Power_W'].corr(df['Torque [Nm]']), 2),
      '-> Power is partly redundant with Torque (fine for trees; note it for linear models).')

Correlation of new features with the target:
Temp_Diff            -0.12
Mechanical_Power_W    0.22
Strain_Index          0.31

Corr(Power, Torque) = 0.93 -> Power is partly redundant with Torque (fine for trees; note it for linear models).


## Step 2 — Feature Selection 

In [3]:
# Evidence 1: UDI separates synthetic rows perfectly -> the model could 'cheat' by reading the row number
print('UDI > 10000 vs Machine failure:')
print(pd.crosstab(df['UDI'] > 10000, df[target]), '\n')

# Evidence 2: Product ID is reused heavily (synthetic rows copy IDs) -> same ID would sit in both train and test
print('Rows:', len(df), '| unique Product IDs:', df['Product ID'].nunique())
print(df['Product ID'].value_counts().head(3).to_string(), '\n')

# Evidence 3: the 5 failure-type columns ARE the target (target = OR of them, ~99.8% of rows)
sub = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
match = ((df[sub].sum(axis=1) > 0).astype(int) == df[target]).mean()
print(f'Machine failure == any(TWF,HDF,PWF,OSF,RNF) in {match*100:.2f}% of rows')

UDI > 10000 vs Machine failure:
Machine failure     0     1
UDI                        
False            9661   339
True                0  2500 

Rows: 12500 | unique Product IDs: 10000
Product ID
H35323    115
L50791     94
M17925     40 

Machine failure == any(TWF,HDF,PWF,OSF,RNF) in 99.78% of rows


In [4]:
drop_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df_sel = df.drop(columns=drop_cols)
print('Columns kept:', df_sel.columns.tolist())
print('Shape:', df_sel.shape)

Columns kept: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'Temp_Diff', 'Mechanical_Power_W', 'Strain_Index']
Shape: (12500, 10)


## Step 3 — Encoding `Type`

In [5]:
type_map = {'L': 0, 'M': 1, 'H': 2}
df_sel['Type'] = df_sel['Type'].map(type_map)
assert df_sel['Type'].isna().sum() == 0, 'Unmapped Type value found!'
print('Encoding map:', type_map)
print(df_sel['Type'].value_counts().sort_index().to_dict())

Encoding map: {'L': 0, 'M': 1, 'H': 2}
{0: 7626, 1: 3589, 2: 1285}


## Step 4 — Train/Test Split 

In [6]:
X = df_sel.drop(columns=[target])
y = df_sel[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_test = X_train.copy(), X_test.copy()    # avoid SettingWithCopyWarning

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print(f'Failure rate  full={y.mean()*100:.2f}%  train={y_train.mean()*100:.2f}%  test={y_test.mean()*100:.2f}%')

# Leak checks BEFORE scaling (raw values still comparable)
idx_overlap = set(X_train.index) & set(X_test.index)
row_overlap = pd.merge(X_train.drop_duplicates(), X_test.drop_duplicates(), how='inner').shape[0]
print('Index overlap train/test      :', len(idx_overlap))
print('Identical feature rows in both:', row_overlap)

X_train: (10000, 9) | X_test: (2500, 9)
Failure rate  full=22.71%  train=22.71%  test=22.72%
Index overlap train/test      : 0
Identical feature rows in both: 0


## Step 5 — Standardization (fit on training set only)

In [7]:
continuous_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
                   'Torque [Nm]', 'Tool wear [min]', 'Temp_Diff', 'Mechanical_Power_W', 'Strain_Index']

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])   # learn from TRAIN only
X_test[continuous_cols]  = scaler.transform(X_test[continuous_cols])        # apply the same numbers

joblib.dump(scaler, 'scaler_milestone3.joblib')   # needed in Module 4 to scale any new data the same way
print('Scaler saved.')

Scaler saved.


## Step 6 — Final Readiness Check 

In [8]:
print('X_train', X_train.shape, '| X_test', X_test.shape)
print(f'Failure rate train={y_train.mean()*100:.2f}% | test={y_test.mean()*100:.2f}%\n')

checks = {
    'All columns numeric'            : all(pd.api.types.is_numeric_dtype(X_train[c]) for c in X_train.columns),
    'No nulls (train+test)'          : X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0,
    'Zero index overlap'             : len(set(X_train.index) & set(X_test.index)) == 0,
    'No leakage/ID columns left'     : not any(c in X_train.columns for c in ['UDI','Product ID','TWF','HDF','PWF','OSF','RNF']),
    'Train mean ~ 0 (scaled cols)'   : bool(np.allclose(X_train[continuous_cols].mean(), 0, atol=1e-6)),
    'Train std  ~ 1 (scaled cols)'   : bool(np.allclose(X_train[continuous_cols].std(ddof=0), 1, atol=1e-6)),
    'Split is stratified (|diff|<0.2%)': abs(y_train.mean() - y_test.mean()) < 0.002,
    'Train+Test = full data'         : len(X_train) + len(X_test) == len(df),
}
for k, v in checks.items(): print(('PASS  ' if v else 'FAIL  ') + k)
assert all(checks.values()), 'Readiness check failed - fix before saving!'

print('\nTest-set mean/std (NOT exactly 0/1 - this is expected and proves we did not peek at the test set):')
print(pd.DataFrame({'mean': X_test[continuous_cols].mean(), 'std': X_test[continuous_cols].std()}).round(3))

X_train (10000, 9) | X_test (2500, 9)
Failure rate train=22.71% | test=22.72%

PASS  All columns numeric
PASS  No nulls (train+test)
PASS  Zero index overlap
PASS  No leakage/ID columns left
PASS  Train mean ~ 0 (scaled cols)
PASS  Train std  ~ 1 (scaled cols)
PASS  Split is stratified (|diff|<0.2%)
PASS  Train+Test = full data

Test-set mean/std (NOT exactly 0/1 - this is expected and proves we did not peek at the test set):
                          mean    std
Air temperature [K]     -0.006  1.002
Process temperature [K]  0.011  1.010
Rotational speed [rpm]  -0.004  0.982
Torque [Nm]             -0.007  1.014
Tool wear [min]          0.001  1.004
Temp_Diff                0.018  0.979
Mechanical_Power_W      -0.014  0.993
Strain_Index            -0.010  0.995


## Before → After Snapshot 

In [9]:
snapshot = pd.DataFrame({
    'Before': [cols_before, f'{len(text_cols_before)} text columns (Product ID, Type)', 'Raw units (var 30,918 vs 2.56)', '0 / 0'],
    'After' : [X_train.shape[1] + 1,
               'Type -> 0/1/2 (ordinal); Product ID dropped',
               'Standardized (mean 0, std 1 on train)',
               f'{len(X_train):,} / {len(X_test):,} (no overlap)'],
}, index=['Columns (incl. target)', 'Category columns', 'Feature scale', 'Rows train / test'])
snapshot

,Before,After
Columns (incl. target),14,10
Category columns,"2 text columns (Product ID, Type)",Type -> 0/1/2 (ordinal); Product ID dropped
Feature scale,"Raw units (var 30,918 vs 2.56)","Standardized (mean 0, std 1 on train)"
Rows train / test,0 / 0,"10,000 / 2,500 (no overlap)"


## Step 7 — Saving Final Files

In [10]:
train_out = pd.concat([X_train, y_train], axis=1)
test_out  = pd.concat([X_test,  y_test],  axis=1)
train_out.to_csv('train_ready_milestone3.csv', index=False)
test_out.to_csv('test_ready_milestone3.csv',  index=False)
print('Saved:', train_out.shape, test_out.shape)
train_out.head()

Saved: (10000, 10) (2500, 10)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temp_Diff,Mechanical_Power_W,Strain_Index,Machine failure
7488,0,0.055696,1.004343,0.135305,0.162788,0.666002,0.893710,0.371931,0.638123,0
6263,1,0.389608,0.316081,1.002262,-1.126202,-1.501036,-0.187093,-0.980471,-1.297848,0
5368,1,1.391342,1.504897,0.826604,-0.911370,0.666002,-0.307182,-0.743445,0.002394,0
10162,2,1.343640,1.379758,0.786940,-1.874135,1.348767,-0.367227,-2.059780,-0.325065,1
11045,0,0.103398,1.817743,-1.819599,2.474219,-0.595630,1.614245,1.659503,0.191126,1
